In [1]:
# 1. Prepare environment requirements
!pip install gradio tensorflow numpy matplotlib


In [7]:


import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, BatchNormalization, ReLU, MaxPooling1D, LSTM, Dense, Dropout
import gradio as gr
import matplotlib.pyplot as plt
import io
from PIL import Image

# -------------------------------------------------------------
# 2. GENERATE ADVANCED MULTI-DIAGNOSIS ARRAYS
# -------------------------------------------------------------
print("Compiling synthetic multi-symptom clinical database...")
np.random.seed(42)
num_samples = 1200
seq_length = 180

X_data = []
y_disease = []  # Targets: 0=Normal, 1=Arrhythmia, 2=Myocardial Infarction (Heart Attack)
y_stress = []   # Targets: 0=Low, 1=High Stress

for _ in range(num_samples):
    disease_type = np.random.choice([0, 1, 2])
    stress_level = np.random.choice([0, 1])

    base_timeline = np.linspace(0, 6 * np.pi, seq_length)

    if disease_type == 0:    # Normal
        wave = np.sin(base_timeline)
    elif disease_type == 1:  # Arrhythmia (High frequency erratic spikes)
        wave = np.sin(np.linspace(0, 14 * np.pi, seq_length)) * 1.2
    else:                    # Myocardial Infarction (Simulating an elevated ST segment)
        wave = np.sin(base_timeline) + 0.6

    if stress_level == 1:
        wave += np.sin(base_timeline * 4) * 0.15

    wave_noisy = wave + np.random.normal(0, 0.05, seq_length)

    X_data.append(wave_noisy)
    y_disease.append(disease_type)
    y_stress.append(stress_level)

X = np.expand_dims(np.array(X_data), axis=-1)
y_disease = tf.keras.utils.to_categorical(y_disease, num_classes=3)
y_stress = tf.keras.utils.to_categorical(y_stress, num_classes=2)

# -------------------------------------------------------------
# 3. MULTI-OUTPUT FUNCTIONAL API DEEP SEQUENCE MODEL
# -------------------------------------------------------------
print("Building Multi-Task Functional Network Backbone...")
input_layer = Input(shape=(seq_length, 1), name="ECG_Telemetry_Input")

x = Conv1D(filters=32, kernel_size=5, padding='same')(input_layer)
x = BatchNormalization()(x)
x = ReLU()(x)
x = MaxPooling1D(pool_size=2)(x)

x = Conv1D(filters=64, kernel_size=5, padding='same')(x)
x = BatchNormalization()(x)
x = ReLU()(x)
x = MaxPooling1D(pool_size=2)(x)

shared_sequence = LSTM(64, return_sequences=False)(x)
shared_sequence = Dropout(0.3)(shared_sequence)

# Target Heads
head_disease = Dense(32, activation='relu')(shared_sequence)
output_disease = Dense(3, activation='softmax', name='Disease_Output')(head_disease)

head_stress = Dense(16, activation='relu')(shared_sequence)
output_stress = Dense(2, activation='softmax', name='Stress_Output')(head_stress)

model = Model(inputs=input_layer, outputs=[output_disease, output_stress])

model.compile(
    optimizer='adam',
    loss={'Disease_Output': 'categorical_crossentropy', 'Stress_Output': 'categorical_crossentropy'},
    metrics={'Disease_Output': 'accuracy', 'Stress_Output': 'accuracy'}
)

print("Training Multi-Task Sequence Target Network...")
model.fit(X, {'Disease_Output': y_disease, 'Stress_Output': y_stress}, epochs=10, batch_size=32, verbose=1)
print("System compiled successfully.")

# -------------------------------------------------------------
# 4. PRE-COMPILING 10 DISTINCT CLINICAL TEST EXAMPLES
# -------------------------------------------------------------
print("Generating 10 labeled clinical examples...")
timeline = np.linspace(0, 6 * np.pi, seq_length)
ten_examples = []

# Base waves configurations mapped out explicitly
waves_config = [
    (0, 0), # 1. Normal, Low Stress
    (0, 1), # 2. Normal, High Stress
    (1, 0), # 3. Arrhythmia, Low Stress
    (1, 1), # 4. Arrhythmia, High Stress
    (2, 0), # 5. Heart Attack Risk, Low Stress
    (2, 1), # 6. Heart Attack Risk, High Stress
    (0, 0), # 7. Variant Normal
    (1, 0), # 8. Variant Arrhythmia
    (2, 1), # 9. Severe Variant Heart Attack + High Stress
    (0, 1)  # 10. Normal baseline with Anxiety noise
]

for idx, (dis, strss) in enumerate(waves_config):
    if dis == 0:
        w = np.sin(timeline)
    elif dis == 1:
        w = np.sin(np.linspace(0, 14 * np.pi, seq_length)) * 1.2
    else:
        w = np.sin(timeline) + 0.6

    if strss == 1:
        w += np.sin(timeline * 4) * 0.15

    # Inject slight randomized variations so examples 7-10 look unique
    w_final = w + np.random.normal(0, 0.04, seq_length)
    formatted_string = ",".join([str(round(x, 4)) for x in w_final])
    ten_examples.append([formatted_string])

# -------------------------------------------------------------
# 5. INFERENCE FUNCTION
# -------------------------------------------------------------
def analyze_advanced_ecg(raw_text_signal):
    try:
        signal_array = np.array([float(x.strip()) for x in raw_text_signal.split(",") if x.strip()])
        if len(signal_array) < seq_length:
            signal_array = np.pad(signal_array, (0, seq_length - len(signal_array)), 'constant')
        else:
            signal_array = signal_array[:seq_length]

        plt.figure(figsize=(6.5, 2.5))
        plt.plot(signal_array, color='#4CAF50', linewidth=2)
        plt.title("Multi-Task Telemetry Waveform Inspection Matrix", fontsize=10)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150)
        buf.seek(0)
        plot_image = Image.open(buf)
        plt.close()

        input_tensor = np.expand_dims(np.expand_dims(signal_array, axis=-1), axis=0)
        disease_preds, stress_preds = model.predict(input_tensor, verbose=0)

        disease_labels = ['Normal Sinus Rhythm', 'Arrhythmia Spikes', 'Myocardial Infarction Risk (Heart Attack)']
        stress_labels = ['Stable / Low Psychological Strain', 'Elevated Autonomic Stress/Anxiety detected']

        selected_disease = disease_labels[np.argmax(disease_preds)]
        selected_stress = stress_labels[np.argmax(stress_preds)]

        diagnostic_report = (
            f"🫀 Primary Cardiac Condition: {selected_disease}\n"
            f"📊 Confidence Match Score: {np.max(disease_preds) * 100:.2f}%\n\n"
            f"🧠 Autonomic Nervous Evaluation: {selected_stress}\n"
            f"📉 Stress Confidence Value: {np.max(stress_preds) * 100:.2f}%"
        )
        return plot_image, diagnostic_report
    except Exception as e:
        return None, f"Parsing Failure. Check data metrics sequence format. Error: {str(e)}"

# Setup Gradio Interface Dashboard with 10 explicit examples
demo = gr.Interface(
    fn=analyze_advanced_ecg,
    inputs=gr.Textbox(lines=4, value=ten_examples[0][0], label="Telematic Multi-Channel Sequence Vector (Comma Separated)"),
    outputs=[gr.Image(type="pil", label="Generated ECG Signal Wave Plot Analysis"), gr.Textbox(label="Multi-Task AI Diagnostic Report")],
    title="导🩺 Next-Gen Multi-Task AI ECG Diagnostic Workstation",
    description="Advanced multi-output Functional API network analyzing cardiac pathology and psychological strain simultaneously.",
    theme=gr.themes.Soft(),
    examples=ten_examples  # Holds all 10 clickable medical rows flawlessly
)

demo.launch(share=True)


Compiling synthetic multi-symptom clinical database...
Building Multi-Task Functional Network Backbone...
Training Multi-Task Sequence Target Network...
Epoch 1/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 8s 40ms/step - Disease_Output_accuracy: 0.9250 - Disease_Output_loss: 0.3981 - Stress_Output_accuracy: 0.5083 - Stress_Output_loss: 0.7066 - loss: 1.1094
Epoch 2/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - Disease_Output_accuracy: 1.0000 - Disease_Output_loss: 0.0191 - Stress_Output_accuracy: 0.5975 - Stress_Output_loss: 0.6583 - loss: 0.6790
Epoch 3/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - Disease_Output_accuracy: 1.0000 - Disease_Output_loss: 0.0087 - Stress_Output_accuracy: 0.7400 - Stress_Output_loss: 0.5390 - loss: 0.5473
Epoch 4/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - Disease_Output_accuracy: 1.0000 - Disease_Output_loss: 0.0105 - Stress_Output_accuracy: 0.9258 - Stress_Output_loss: 0.2733 - loss: 0.2858
Epoch 5/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - Disease_Output_accuracy: 1

/usr/local/lib/python3.13/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d63fa1f19d935773d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
# Save the complete multi-output network architecture and weights
model.save("ecg_multitask_backbone.keras")
print("Model successfully serialized and ready for deployment storage.")


Model successfully serialized and ready for deployment storage.
